# Ant Colony Optimization Lab: Python companion
Interactive companion to Chapter 10 of [*Optimization Algorithms: AI techniques for design, planning, and control problems*](https://www.manning.com/books/optimization-algorithms) by Alaa Khamis (Manning, 2024).

These from-scratch implementations mirror the simulations in the ACO Lab. They use only NumPy and Matplotlib, which are preinstalled in Google Colab.

## 1. Simple ACO on the shortest path problem
The graph of Figures 10.7 to 10.10: reproduce the transition probabilities of equations 10.7 and 10.8, then let the colony converge.

In [ ]:
# Simple ACO (SACO) on the shortest path problem of Chapter 10 (Figures 10.7 to 10.10)
import numpy as np

# edge lengths of the book's graph: source node 1, goal node 5
edges = {(1, 2): 4, (1, 3): 3, (2, 3): 5, (2, 4): 4, (3, 4): 2, (2, 5): 1, (4, 5): 6}
D = {}
for (a, b), d in edges.items():
    D[a, b] = D[b, a] = d
nbrs = {n: sorted(m for (k, m) in D if k == n) for n in range(1, 6)}


def transition(i, tau, visited, alpha=1.0, beta=1.0):
    """Equation 10.6: p_ij = tau_ij^alpha * eta_ij^beta / sum over feasible j, eta = 1/d."""
    cand = [j for j in nbrs[i] if j not in visited]
    w = np.array([tau[i, j] ** alpha * (1 / D[i, j]) ** beta for j in cand])
    return cand, w / w.sum()


tau = {e: 0.5 for e in D}
print("from node 1:", dict(zip(*transition(1, tau, {1}))))        # p12 = 0.43, p13 = 0.57
print("from node 3:", dict(zip(*transition(3, tau, {1, 3}))))     # p32 = 0.29, p34 = 0.71
print("from node 4:", dict(zip(*transition(4, tau, {1, 3, 4}))))  # p42 = 0.60, p45 = 0.40


def saco(ants=5, iterations=5, tau0=0.5, rho=0.7, alpha=1.0, beta=1.0, Q=1.0, seed=0):
    rng = np.random.default_rng(seed)
    tau = {e: tau0 for e in D}
    best_path, best_len = None, np.inf
    for it in range(iterations):
        paths = []
        for k in range(ants):
            path = [1]
            while path[-1] != 5:                       # construct a path to the goal node
                cand, p = transition(path[-1], tau, set(path), alpha, beta)
                if not cand:                           # dead end: start again
                    path = [1]
                    continue
                path.append(cand[rng.choice(len(cand), p=p)])
            paths.append(path)
        for e in tau:                                  # evaporation (eq. 10.2)
            tau[e] *= (1 - rho)
        for path in paths:                             # online delayed update (eq. 10.4)
            L = sum(D[a, b] for a, b in zip(path, path[1:]))
            for a, b in zip(path, path[1:]):
                tau[a, b] += Q / L
                tau[b, a] += Q / L
            if L < best_len:
                best_path, best_len = path, L
        print(f"iteration {it + 1}: paths {[''.join(map(str, p)) for p in paths]}")
    return best_path, best_len, tau


best_path, best_len, tau = saco(seed=1)
print("best path:", " -> ".join(map(str, best_path)), "length", best_len)
print("pheromone on 1-2-5:", round(tau[1, 2], 3), round(tau[2, 5], 3),
      " on 1-3-4-5:", round(tau[1, 3], 3), round(tau[3, 4], 3), round(tau[4, 5], 3))

## 2. Open TSP with AS, ACS and MMAS
The 20 US cities of Listing 10.1, starting from New York City without returning. The exact optimum for this instance is 7,909.2 km.

In [ ]:
# Open TSP with ACO variants (Chapter 10, Section 10.3.5): AS, ACS and MMAS
# 20 major US cities, start at New York City, no return to the home city.
import numpy as np
import matplotlib.pyplot as plt

cities = {
    "New York City": (40.72, -74.00), "Philadelphia": (39.95, -75.17),
    "Baltimore": (39.28, -76.62), "Charlotte": (35.23, -80.85),
    "Memphis": (35.12, -89.97), "Jacksonville": (30.32, -81.70),
    "Houston": (29.77, -95.38), "Austin": (30.27, -97.77),
    "San Antonio": (29.53, -98.47), "Fort Worth": (32.75, -97.33),
    "Dallas": (32.78, -96.80), "San Diego": (32.78, -117.15),
    "Los Angeles": (34.05, -118.25), "San Jose": (37.30, -121.87),
    "San Francisco": (37.78, -122.42), "Indianapolis": (39.78, -86.15),
    "Phoenix": (33.45, -112.07), "Columbus": (39.98, -82.98),
    "Chicago": (41.88, -87.63), "Detroit": (42.33, -83.05),
}
names = list(cities)
lat, lon = np.radians(np.array(list(cities.values()))).T
a = (np.sin((lat[:, None] - lat) / 2) ** 2
     + np.cos(lat[:, None]) * np.cos(lat) * np.sin((lon[:, None] - lon) / 2) ** 2)
D = 2 * 6371.0088 * np.arcsin(np.sqrt(a))          # haversine distances in km
n = len(names)
eta = 1 / (D + np.eye(n))                           # desirability 1/d (diagonal unused)
length = lambda p: D[p[:-1], p[1:]].sum()           # open path: no return to the start


def run_aco(variant="AS", ants=30, iterations=100, alpha=1.0, beta=0.9, rho=0.5, Q=100,
            q0=0.9, xi=0.1, seed=0):
    rng = np.random.default_rng(seed)
    tau0 = 1.0 if variant == "AS" else 1 / (n * 8000)
    tau = np.full((n, n), tau0)
    if variant == "MMAS":
        tau_max = 1 / (rho * 8000); tau[:] = tau_max
    best_p, best_L, history = None, np.inf, []
    for it in range(iterations):
        paths = []
        for k in range(ants):
            p, free = [0], np.ones(n, bool); free[0] = False
            while free.any():
                i, cand = p[-1], np.flatnonzero(free)
                w = tau[i, cand] ** alpha * eta[i, cand] ** beta
                if variant == "ACS" and rng.random() <= q0:      # exploitation (eq. 10.10)
                    j = cand[np.argmax(w)]
                else:                                            # biased exploration (eq. 10.6)
                    j = rng.choice(cand, p=w / w.sum())
                if variant == "ACS":                             # ACS local update
                    tau[i, j] = tau[j, i] = (1 - xi) * tau[i, j] + xi * tau0
                p.append(j); free[j] = False
            paths.append(np.array(p))
        Ls = np.array([length(p) for p in paths])
        b = Ls.argmin()
        if Ls[b] < best_L:
            best_p, best_L = paths[b], Ls[b]
        if variant == "AS":                                      # every ant deposits Q / L
            tau *= (1 - rho)
            for p, L in zip(paths, Ls):
                tau[p[:-1], p[1:]] += Q / L; tau[p[1:], p[:-1]] += Q / L
        elif variant == "ACS":                                   # only the global-best ant
            e = (best_p[:-1], best_p[1:])
            tau[e] = (1 - rho) * tau[e] + rho / best_L
            tau[e[::-1]] = tau[e]
        else:                                                    # MMAS: iteration-best, bounded
            tau *= (1 - rho)
            p = paths[b]
            tau[p[:-1], p[1:]] += 1 / Ls[b]; tau[p[1:], p[:-1]] += 1 / Ls[b]
            tau_max = 1 / (rho * best_L); tau_min = tau_max / (2 * n)
            np.clip(tau, tau_min, tau_max, out=tau)
        history.append(best_L)
    return best_p, best_L, history


for variant in ["AS", "ACS", "MMAS"]:
    p, L, hist = run_aco(variant, seed=1)
    print(f"{variant:5s} {L:9.1f} km  " + " -> ".join(names[i] for i in p))
    plt.plot(hist, label=variant)
plt.axhline(7937.115, ls="--", c="gray", label="book's run (7,937 km)")
plt.xlabel("iteration"); plt.ylabel("best open-route length (km)"); plt.legend(); plt.show()

## 3. ACO for a continuous variable
Minimize f(x) = x² − 2x − 11 on [0, 3] by discretizing x into 7 paths. The first two iterations use the random numbers of the worked example.

In [ ]:
# ACO for continuous optimization: minimize f(x) = x^2 - 2x - 11 on [0, 3]
# The range is discretized into 7 values; each value is a path from home to food.
import numpy as np

f = lambda x: x**2 - 2 * x - 11
xs = np.arange(0, 3.01, 0.5)                    # x11 ... x17 = 0.0, 0.5, ..., 3.0


def aco_continuous(randoms=None, ants=4, iterations=3, rho=0.5, zeta=2.0, seed=0):
    rng = np.random.default_rng(seed)
    tau = np.ones(len(xs))                      # equal pheromone on every path
    for it in range(iterations):
        p = tau / tau.sum()                     # selection probabilities
        cum = np.cumsum(p)
        r = randoms[it] if randoms and it < len(randoms) else rng.random(ants)
        chosen = np.searchsorted(cum, r)        # roulette wheel
        fx = f(xs[chosen])
        best, worst = fx.min(), fx.max()
        n_best = np.sum(fx == best)
        tau *= (1 - rho)                        # evaporation
        tau[chosen[fx == best][0]] += n_best * zeta * best / worst   # best ants deposit
        print(f"iteration {it + 1}: x = {xs[chosen]}, f = {fx}, tau = {np.round(tau, 4)}")
    return xs[np.argmax(tau)]


# the worked example's random numbers for iterations 1 and 2
x_best = aco_continuous(randoms=[[0.3122, 0.8701, 0.4729, 0.6190], [0.3688, 0.8577, 0.0776, 0.5791]])
print("most reinforced path: x =", x_best, " f =", f(x_best))   # x = 1.0, f = -12